In [5]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/final.csv")


In [6]:
pip install kneed

In [8]:
"""
lstm_cluster_pipeline_v5.py — Transformer Context + FiLM-Conditioned LSTM Volatility
======================================================================================
v5 change: Cluster assignments are hardcoded (no profiling / KMeans step).

Hardcoded clusters
------------------
  Cluster 0 (38 stocks): [0, 3, 4, 5, 6, 9, 11, 16, 18, 22, 27, 30, 33, 37,
      38, 40, 55, 62, 75, 78, 80, 81, 83, 87, 88, 90, 94, 97, 98, 100, 102,
      103, 110, 112, 114, 116, 118, 126]
  Cluster 1 (69 stocks): [1, 2, 7, 8, 10, 13, 14, 15, 17, 19, 20, 21, 23, 26,
      28, 29, 34, 35, 36, 39, 42, 43, 44, 46, 47, 48, 50, 51, 52, 53, 56, 58,
      59, 60, 61, 63, 64, 66, 67, 68, 69, 70, 72, 73, 74, 76, 82, 84, 85, 86,
      89, 93, 95, 96, 99, 101, 104, 105, 107, 109, 111, 113, 115, 119, 120,
      122, 123, 124, 125]
  Cluster 2 (5 stocks):  [31, 32, 41, 77, 108]

Architecture (unchanged from v4)
---------------------------------
  480s WAP  →  10s buckets (48 steps)
               3 features per bucket: [ret_bucket, log_wap_bucket, rv_bucket]
                            ↓
               ┌──────────────────────────────┐
               │  Transformer Context Branch  │
               │  Linear(3 → context_dim)     │
               │  + learned positional enc    │
               │  TransformerEncoder          │
               │    n_layers=2, n_heads=4     │
               │    FFN dim = context_dim × 4 │
               │    pre-norm (norm_first=True) │
               │  global avg-pool             │
               └──────────┬───────────────────┘
                          │  → (B*K, context_dim=64)
                          │
                    FiLM layer
                    γ, β = Linear(64, 256) each
                          │
  60s WAP   →  [ret, log_wap]
               ↓
               ┌────────────────────────┐
               │   Local LSTM Branch    │
               │   3 layers × 256 units │
               │   dropout = 0.2        │
               └──────────┬─────────────┘
                          │  h_last → (B*K, 256)
                          │
                    FiLM: γ · h + β    ← context modulates LSTM repr
                          │
                    Linear(256, 64) → ReLU → Linear(64, 1) → (B, K)
"""

import warnings
import time as _time

import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────
# HARDCODED CLUSTER ASSIGNMENTS
# ─────────────────────────────────────────────────────────────────────

HARDCODED_CLUSTERS: dict[int, list[int]] = {
    0: [0, 3, 4, 5, 6, 9, 11, 16, 18, 22, 27, 30, 33, 37, 38, 40, 55, 62,
        75, 78, 80, 81, 83, 87, 88, 90, 94, 97, 98, 100, 102, 103, 110, 112,
        114, 116, 118, 126],
    1: [1, 2, 7, 8, 10, 13, 14, 15, 17, 19, 20, 21, 23, 26, 28, 29, 34, 35,
        36, 39, 42, 43, 44, 46, 47, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61,
        63, 64, 66, 67, 68, 69, 70, 72, 73, 74, 76, 82, 84, 85, 86, 89, 93,
        95, 96, 99, 101, 104, 105, 107, 109, 111, 113, 115, 119, 120, 122,
        123, 124, 125],
    2: [31, 32, 41, 77, 108],
}


def build_cluster_map(stock_cols: list) -> pd.Series:
    """
    Convert HARDCODED_CLUSTERS (column-name → cluster-id) into a
    pd.Series keyed by stock column name, as expected by the dataset.

    The values in HARDCODED_CLUSTERS are column names (e.g. 0, 1, 2 …),
    not positional indices into stock_cols.
    """
    stock_col_set = set(stock_cols)
    mapping: dict = {}
    skipped = []
    for cid, names in HARDCODED_CLUSTERS.items():
        for name in names:
            if name in stock_col_set:
                mapping[name] = cid
            else:
                skipped.append(name)
    if skipped:
        warnings.warn(
            f"{len(skipped)} name(s) in HARDCODED_CLUSTERS not found in "
            f"stock_cols and will be ignored: {skipped}"
        )

    # Any stock not mentioned is assigned cluster -1 (safety net)
    unclaimed = [c for c in stock_cols if c not in mapping]
    if unclaimed:
        warnings.warn(
            f"{len(unclaimed)} stock column(s) not in any hardcoded cluster "
            f"and will be assigned cluster id -1: {unclaimed}"
        )
        for col in unclaimed:
            mapping[col] = -1

    cluster_map = pd.Series(mapping, name="cluster_id")
    print("Hardcoded cluster sizes:")
    for cid, cnt in cluster_map.value_counts().sort_index().items():
        print(f"  Cluster {cid}: {cnt} stocks")
    return cluster_map


# ─────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────

CONTEXT_LEN  = 480
LOCAL_LEN    = 60
TARGET_LEN   = 120
BUCKET_SIZE  = 10
N_BUCKETS    = CONTEXT_LEN // BUCKET_SIZE   # 48
TOTAL_LEN    = 600
STRIDE       = CONTEXT_LEN + TARGET_LEN     # 600

EPSILON      = 1e-5
RET_CLIP     = 0.05

CONTEXT_DIM         = 64
TRANSFORMER_HEADS   = 4
TRANSFORMER_LAYERS  = 2
TRANSFORMER_DROPOUT = 0.1

N_CHANNELS      = 2
LSTM_HIDDEN     = 256
LSTM_LAYERS     = 3
LSTM_DROPOUT    = 0.2

TCN_IN_CHANNELS = 3
AUX_ALPHA       = 0.2

SAMPLES_PER_TID = max(1, TOTAL_LEN // STRIDE)


# ─────────────────────────────────────────────────────────────────────
# LOW-LEVEL FEATURE HELPERS
# ─────────────────────────────────────────────────────────────────────

def _safe_log_return_matrix(mat: np.ndarray) -> np.ndarray:
    safe = np.where(mat > 0, mat, 1e-6).astype(np.float32)
    ret  = np.zeros_like(safe)
    ret[1:] = np.log(safe[1:] / safe[:-1])
    np.clip(ret, -RET_CLIP, RET_CLIP, out=ret)
    np.nan_to_num(ret, copy=False, nan=0.0, posinf=RET_CLIP, neginf=-RET_CLIP)
    return ret


def robust_fill(mat: np.ndarray) -> np.ndarray:
    bad = ~np.isfinite(mat) | (mat <= 0)
    if not bad.any():
        return mat
    out     = mat.copy()
    rows    = out.shape[0]
    row_idx = np.arange(rows)[:, None]
    fwd     = np.where(~bad, row_idx, -1)
    np.maximum.accumulate(fwd, axis=0, out=fwd)
    rev = np.where(~bad, row_idx, rows)
    bwd = rows - 1 - np.maximum.accumulate(
        (rows - 1 - rev)[::-1], axis=0
    )[::-1]
    use     = np.where(fwd >= 0, fwd, bwd)
    all_bad = (use < 0) | (use >= rows)
    use     = np.where(all_bad, 0, use)
    out     = out[use, np.arange(out.shape[1])]
    return np.where(all_bad, 1.0, out)


def compute_local_features(wap_mat: np.ndarray,
                            open_prices: np.ndarray) -> np.ndarray:
    """(T, S) WAP → (T, S, 2)  [return, log_wap]."""
    ret      = _safe_log_return_matrix(wap_mat)
    safe_wap = np.maximum(wap_mat, 1e-6).astype(np.float32)
    log_wap  = np.log(safe_wap / (open_prices[None, :] + EPSILON)).astype(np.float32)
    feat = np.stack([ret, log_wap], axis=-1)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return feat


def bucket_context_features(wap_ctx: np.ndarray,
                             open_price_ctx: np.ndarray,
                             bucket_size: int = BUCKET_SIZE) -> np.ndarray:
    """
    Aggregate 480s WAP into N_BUCKETS=48 coarse timesteps.

    Returns
    -------
    bucket_feat : (N_BUCKETS=48, S, 3)
        channel 0 : ret_bucket   — sum of log-returns within bucket
        channel 1 : log_wap      — log(close_of_bucket / context_open)
        channel 2 : rv_bucket    — sqrt(sum r^2) within bucket
    """
    T, S = wap_ctx.shape
    n_b  = T // bucket_size
    ret  = _safe_log_return_matrix(wap_ctx)

    ret_b  = ret[:n_b * bucket_size].reshape(n_b, bucket_size, S)
    wap_b  = wap_ctx[:n_b * bucket_size].reshape(n_b, bucket_size, S)

    ret_sum   = ret_b.sum(axis=1)
    rv_b      = np.sqrt((ret_b ** 2).sum(axis=1) + EPSILON)
    wap_close = wap_b[:, -1, :]
    log_wap   = np.log(
        np.maximum(wap_close, 1e-6) / (open_price_ctx[None, :] + EPSILON)
    ).astype(np.float32)

    feat = np.stack([ret_sum, log_wap, rv_b], axis=-1)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return feat.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────
# SPLIT
# ─────────────────────────────────────────────────────────────────────

def make_splits_from_time_id(df: pd.DataFrame,
                              val_ratio:  float = 0.15,
                              test_ratio: float = 0.15,
                              seed:       int   = 42) -> dict:
    rng    = np.random.default_rng(seed)
    tids   = df["time_id"].unique().copy()
    rng.shuffle(tids)
    n      = len(tids)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)
    return {"split_summary": {
        "train": {"ids": tids[n_test + n_val:]},
        "val":   {"ids": tids[n_test : n_test + n_val]},
        "test":  {"ids": tids[:n_test]},
    }}


# ─────────────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────────────

class TCNDualBranchDataset(Dataset):
    """
    Per sample yields:
        x_local   : (K, LOCAL_LEN=60, 2)
        x_tcn     : (K, N_BUCKETS=48, 3)
        y_main    : (K,)  log(RV_fut / RV_local)
        y_aux     : (K,)  log(RV_480 / RV_60)
        rv_inputs : (K,)  raw RV_local for retransformation
    """

    def __init__(self, df: pd.DataFrame, stock_cols: list,
                 cluster_map: pd.Series, time_ids: np.ndarray,
                 name: str = ""):
        self.stock_cols  = stock_cols
        cluster_ids      = sorted(cluster_map.unique())
        self.cluster_ids = cluster_ids
        self.n_clusters  = len(cluster_ids)

        self.cluster_col_idx = {
            cid: np.array(
                [stock_cols.index(s)
                 for s in cluster_map.index[cluster_map == cid]],
                dtype=np.int32,
            )
            for cid in cluster_ids
        }

        df = df[df["time_id"].isin(time_ids)].sort_values(
            ["time_id", "seconds_in_bucket"]
        )

        self._samples    = []
        self._wap_blocks = {}
        for tid, grp in df.groupby("time_id", sort=True):
            mat = robust_fill(grp[stock_cols].values.astype(np.float32))
            if mat.shape[0] < TOTAL_LEN:
                continue
            self._wap_blocks[tid] = mat
            for offset in range(0, TOTAL_LEN - CONTEXT_LEN - TARGET_LEN + 1, STRIDE):
                self._samples.append((tid, offset))

        n_tids = len(self._wap_blocks)
        mb = sum(v.nbytes for v in self._wap_blocks.values()) / 1e6
        print(f"  [{name:5s}] {n_tids:>5} time_ids × {SAMPLES_PER_TID} windows "
              f"= {len(self._samples)} samples | cache {mb:.1f} MB")

    def __len__(self) -> int:
        return len(self._samples)

    def __getitem__(self, idx):
        tid, offset = self._samples[idx]
        mat = self._wap_blocks[tid]

        ctx_start = offset
        ctx_end   = offset + CONTEXT_LEN
        loc_start = ctx_end - LOCAL_LEN
        loc_end   = ctx_end
        tgt_start = ctx_end
        tgt_end   = ctx_end + TARGET_LEN

        mat_ctx   = mat[ctx_start : ctx_end]
        mat_local = mat[loc_start : loc_end]
        mat_tgt   = mat[tgt_start : tgt_end]

        open_local = mat_local[0]
        open_ctx   = mat_ctx[0]

        feat_local  = compute_local_features(mat_local, open_local)
        ret_local   = _safe_log_return_matrix(mat_local)
        bucket_feat = bucket_context_features(mat_ctx, open_ctx)
        ret_ctx     = _safe_log_return_matrix(mat_ctx)
        ret_fut     = _safe_log_return_matrix(mat_tgt)

        x_local   = np.empty((self.n_clusters, LOCAL_LEN, N_CHANNELS), dtype=np.float32)
        x_tcn     = np.empty((self.n_clusters, N_BUCKETS, TCN_IN_CHANNELS), dtype=np.float32)
        y_main    = np.empty(self.n_clusters,  dtype=np.float32)
        y_aux     = np.empty(self.n_clusters,  dtype=np.float32)
        rv_inputs = np.empty(self.n_clusters,  dtype=np.float32)

        for ci, cid in enumerate(self.cluster_ids):
            idx_c = self.cluster_col_idx[cid]

            x_local[ci] = feat_local[:, idx_c, :].mean(axis=1)
            x_tcn[ci]   = bucket_feat[:, idx_c, :].mean(axis=1)

            r_in          = ret_local[:, idx_c]
            rv_in         = float(np.sqrt(
                np.einsum("ij,ij->j", r_in, r_in)).mean()) + EPSILON
            rv_inputs[ci] = rv_in

            r_fut         = ret_fut[:, idx_c]
            rv_fut        = float(np.sqrt(
                np.einsum("ij,ij->j", r_fut, r_fut)).mean()) + EPSILON
            y_main[ci]    = np.log(rv_fut / rv_in)

            r_ctx         = ret_ctx[:, idx_c]
            rv_ctx_full   = float(np.sqrt(
                np.einsum("ij,ij->j", r_ctx, r_ctx)).mean()) + EPSILON
            y_aux[ci]     = float(np.log(rv_ctx_full / rv_in))

        np.nan_to_num(x_local,   copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.nan_to_num(x_tcn,     copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.nan_to_num(y_aux,     copy=False, nan=0.0, posinf=0.0, neginf=0.0)

        return (
            torch.from_numpy(x_local),
            torch.from_numpy(x_tcn),
            torch.from_numpy(y_main),
            torch.from_numpy(y_aux),
            torch.from_numpy(rv_inputs),
        )


# ─────────────────────────────────────────────────────────────────────
# TRANSFORMER CONTEXT ENCODER
# ─────────────────────────────────────────────────────────────────────

class TransformerContextEncoder(nn.Module):
    def __init__(self,
                 in_channels:  int   = TCN_IN_CHANNELS,
                 context_dim:  int   = CONTEXT_DIM,
                 n_heads:      int   = TRANSFORMER_HEADS,
                 n_layers:     int   = TRANSFORMER_LAYERS,
                 dropout:      float = TRANSFORMER_DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, context_dim)
        self.pos_enc    = nn.Parameter(
            torch.randn(1, N_BUCKETS, context_dim) * 0.02
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = context_dim,
            nhead           = n_heads,
            dim_feedforward = context_dim * 4,
            dropout         = dropout,
            activation      = "gelu",
            batch_first     = True,
            norm_first      = True,
        )
        self.encoder  = nn.TransformerEncoder(
            encoder_layer,
            num_layers           = n_layers,
            enable_nested_tensor = False,
        )
        self.out_proj = nn.Linear(context_dim, context_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x) + self.pos_enc
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.out_proj(x)


# ─────────────────────────────────────────────────────────────────────
# FiLM CONDITIONING LAYER
# ─────────────────────────────────────────────────────────────────────

class FiLM(nn.Module):
    def __init__(self, context_dim: int, feature_dim: int):
        super().__init__()
        self.gamma_proj = nn.Linear(context_dim, feature_dim)
        self.beta_proj  = nn.Linear(context_dim, feature_dim)
        nn.init.zeros_(self.gamma_proj.weight)
        nn.init.ones_(self.gamma_proj.bias)
        nn.init.zeros_(self.beta_proj.weight)
        nn.init.zeros_(self.beta_proj.bias)

    def forward(self, h: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        return self.gamma_proj(context) * h + self.beta_proj(context)


# ─────────────────────────────────────────────────────────────────────
# FULL MODEL
# ─────────────────────────────────────────────────────────────────────

class TransformerFiLMVolatilityModel(nn.Module):
    def __init__(
        self,
        n_clusters:           int   = 3,
        hidden_dim:           int   = LSTM_HIDDEN,
        num_layers:           int   = LSTM_LAYERS,
        dropout:              float = LSTM_DROPOUT,
        context_dim:          int   = CONTEXT_DIM,
        transformer_heads:    int   = TRANSFORMER_HEADS,
        transformer_layers:   int   = TRANSFORMER_LAYERS,
        transformer_dropout:  float = TRANSFORMER_DROPOUT,
    ):
        super().__init__()
        self.n_clusters = n_clusters

        self.lstm = nn.LSTM(
            input_size  = N_CHANNELS,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout,
        )
        self.lstm_drop  = nn.Dropout(dropout)
        self.transformer = TransformerContextEncoder(
            in_channels = TCN_IN_CHANNELS,
            context_dim = context_dim,
            n_heads     = transformer_heads,
            n_layers    = transformer_layers,
            dropout     = transformer_dropout,
        )
        self.film     = FiLM(context_dim=context_dim, feature_dim=hidden_dim)
        self.head     = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.aux_head = nn.Linear(context_dim, 1)

    def forward(self, x_local: torch.Tensor, x_ctx: torch.Tensor) -> tuple:
        B, K, T, F   = x_local.shape
        _, _, Tb, Fb = x_ctx.shape

        ctx_flat  = x_ctx.reshape(B * K, Tb, Fb)
        ctx_repr  = self.transformer(ctx_flat)

        x_flat         = x_local.reshape(B * K, T, F)
        _, (h_n, _)    = self.lstm(x_flat)
        h_last         = self.lstm_drop(h_n[-1])

        h_cond    = self.film(h_last, ctx_repr)
        pred_main = self.head(h_cond).squeeze(1).reshape(B, K)
        pred_aux  = self.aux_head(ctx_repr).squeeze(1).reshape(B, K)

        return pred_main, pred_aux


# ─────────────────────────────────────────────────────────────────────
# LOSS
# ─────────────────────────────────────────────────────────────────────

class QLIKELoss(nn.Module):
    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        eps = y_true - y_pred
        return (torch.exp(eps) - eps - 1.0).mean()


def get_loss_fn(loss_type: str = "qlike") -> nn.Module:
    return QLIKELoss() if loss_type == "qlike" else nn.MSELoss()


class DualLoss(nn.Module):
    def __init__(self, main_loss: nn.Module, alpha: float = AUX_ALPHA):
        super().__init__()
        self.main_loss = main_loss
        self.aux_loss  = nn.MSELoss()
        self.alpha     = alpha

    def forward(self, pred_main, y_main, pred_aux, y_aux) -> tuple:
        l_main = self.main_loss(pred_main, y_main)
        l_aux  = self.aux_loss(pred_aux, y_aux)
        return l_main + self.alpha * l_aux, l_main, l_aux


# ─────────────────────────────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────────────────────────────

def train_model(model:        nn.Module,
                train_loader: DataLoader,
                val_loader:   DataLoader,
                epochs:       int   = 50,
                lr:           float = 1e-3,
                patience:     int   = 10,
                loss_type:    str   = "qlike",
                ckpt_path:    str   = "best_transformer_lstm.pt") -> nn.Module:

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model   = model.to(device)

    ctx_params  = (list(model.transformer.parameters()) +
                   list(model.film.parameters()) +
                   list(model.aux_head.parameters()))
    lstm_params = (list(model.lstm.parameters()) +
                   list(model.lstm_drop.parameters()) +
                   list(model.head.parameters()))

    opt = torch.optim.Adam([
        {"params": lstm_params, "lr": lr},
        {"params": ctx_params,  "lr": lr * 2.0},
    ], weight_decay=1e-5)

    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=5
    )
    loss_fn = DualLoss(get_loss_fn(loss_type), alpha=AUX_ALPHA)
    print(f"  Loss: {loss_type.upper()} + {AUX_ALPHA}×MSE(aux) | Device: {device}")

    best, stale = float("inf"), 0

    for epoch in range(1, epochs + 1):
        t0 = _time.time()

        model.train()
        tr, n = 0.0, 0
        for x_local, x_tcn, y_main, y_aux, _ in train_loader:
            x_local, x_tcn = x_local.to(device), x_tcn.to(device)
            y_main,  y_aux = y_main.to(device),  y_aux.to(device)
            opt.zero_grad()
            pm, pa = model(x_local, x_tcn)
            loss, _, _ = loss_fn(pm, y_main, pa, y_aux)
            if not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            tr += loss.item() * x_local.size(0)
            n  += x_local.size(0)
        tr /= max(n, 1)

        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for x_local, x_tcn, y_main, y_aux, _ in val_loader:
                x_local, x_tcn = x_local.to(device), x_tcn.to(device)
                y_main,  y_aux = y_main.to(device),  y_aux.to(device)
                pm, pa         = model(x_local, x_tcn)
                l, _, _        = loss_fn(pm, y_main, pa, y_aux)
                vl += l.item() * x_local.size(0)
                n  += x_local.size(0)
        vl /= max(n, 1)

        sched.step(vl)
        improved = (best - vl) > 0.001
        tag = ""
        if improved:
            best, stale = vl, 0
            torch.save(model.state_dict(), ckpt_path)
            tag = " ★"
        else:
            stale += 1

        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f"  Epoch {epoch:>3}/{epochs} | Train {tr:.6f} | "
                  f"Val {vl:.6f} | {_time.time()-t0:.1f}s{tag}")

        if stale >= patience:
            print(f"  Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    return model


# ─────────────────────────────────────────────────────────────────────
# SMEARING
# ─────────────────────────────────────────────────────────────────────

def compute_smearing_factor(model: nn.Module,
                             loader: DataLoader) -> np.ndarray:
    device = next(model.parameters()).device
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for x_local, x_tcn, y_main, _, _ in loader:
            pm, _ = model(x_local.to(device), x_tcn.to(device))
            preds.append(pm.cpu().numpy())
            actuals.append(y_main.numpy())
    preds   = np.concatenate(preds)
    actuals = np.concatenate(actuals)
    delta   = np.mean(np.exp(actuals - preds), axis=0)
    print(f"  Per-cluster smearing δ̂: {np.round(delta, 4).tolist()}")
    return delta


# ─────────────────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────────────────

def _rv_metrics(rv_pred: np.ndarray, rv_act: np.ndarray) -> dict:
    fp    = np.maximum(rv_pred.ravel(), EPSILON)
    fa    = np.maximum(rv_act.ravel(),  EPSILON)
    mse   = mean_squared_error(fa, fp)
    pct   = (fp - fa) / fa
    ratio = fa / (fp + EPSILON)
    return dict(
        mse   = float(mse),
        rmse  = float(np.sqrt(mse)),
        r2    = float(r2_score(fa, fp)),
        mae   = float(np.mean(np.abs(fp - fa))),
        rmspe = float(np.sqrt(np.mean(pct ** 2))),
        mape  = float(np.mean(np.abs(pct))),
        qlike = float(np.mean(ratio - np.log(ratio + EPSILON) - 1)),
    )


def _print_metrics(m: dict, label: str) -> None:
    w = 48
    print(f"\n  ┌{'─' * w}┐")
    print(f"  │  {label:<{w - 2}}│")
    print(f"  ├{'─' * w}┤")
    print(f"  │  MSE   : {m['mse']:<12.8f}   RMSE  : {m['rmse']:<12.8f}│")
    print(f"  │  R²    : {m['r2']:<12.6f}   MAE   : {m['mae']:<12.8f}│")
    print(f"  │  RMSPE : {m['rmspe']:<12.6f}   MAPE  : {m['mape']:<12.6f}│")
    print(f"  │  QLIKE : {m['qlike']:<12.6f}{'':>22}│")
    print(f"  └{'─' * w}┘")


def evaluate_model(model:    nn.Module,
                   loader:   DataLoader,
                   smearing: np.ndarray,
                   label:    str = "") -> dict:
    device = next(model.parameters()).device
    model.eval()
    logp_list, logy_list, rvin_list = [], [], []

    with torch.no_grad():
        for x_local, x_tcn, y_main, _, rv_in in loader:
            pm, _ = model(x_local.to(device), x_tcn.to(device))
            logp_list.append(pm.cpu().numpy())
            logy_list.append(y_main.numpy())
            rvin_list.append(rv_in.numpy())

    logp = np.concatenate(logp_list)
    logy = np.concatenate(logy_list)
    rvin = np.concatenate(rvin_list)

    rv_act = np.exp(logy) * rvin
    rv_raw = np.exp(logp) * rvin
    m_raw  = _rv_metrics(rv_raw, rv_act)
    rv_sme = np.exp(logp) * smearing[None, :] * rvin
    m_sme  = _rv_metrics(rv_sme, rv_act)

    if label:
        print(f"\n{'═' * 52}")
        print(f"  {label}")
        print(f"{'═' * 52}")

    _print_metrics(m_raw, "No Smearing   — exp(ŷ) × RV_input")
    _print_metrics(m_sme, "Smearing-Corrected — exp(ŷ) × δ̂ × RV_input")
    return {"no_smearing": m_raw, "smearing": m_sme}


# ─────────────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────

def run_transformer_film_pipeline(
    df:                   pd.DataFrame,
    stock_cols:           list,
    splits:               dict,
    seed:                 int   = 42,
    hidden_dim:           int   = LSTM_HIDDEN,
    num_layers:           int   = LSTM_LAYERS,
    lstm_dropout:         float = LSTM_DROPOUT,
    context_dim:          int   = CONTEXT_DIM,
    transformer_heads:    int   = TRANSFORMER_HEADS,
    transformer_layers:   int   = TRANSFORMER_LAYERS,
    transformer_dropout:  float = TRANSFORMER_DROPOUT,
    epochs:               int   = 50,
    bs:                   int   = 32,
    lr:                   float = 1e-3,
    patience:             int   = 10,
    loss_type:            str   = "qlike",
    workers:              int   = 0,
    pretrained_path:      str   = None,
    ckpt_path:            str   = "best_transformer_lstm.pt",
) -> dict:

    n_clusters = len(HARDCODED_CLUSTERS)

    print("=" * 66)
    print("Transformer-FiLM + Local LSTM Volatility Pipeline  [v5]")
    print(f"  Clusters  : {n_clusters} (hardcoded — no profiling/KMeans)")
    for cid, idxs in HARDCODED_CLUSTERS.items():
        print(f"    Cluster {cid}: {len(idxs)} stocks")
    print(f"  Context   : {CONTEXT_LEN}s → {N_BUCKETS} buckets of {BUCKET_SIZE}s each")
    print(f"  Transformer: layers={transformer_layers}, heads={transformer_heads}, "
          f"d_model={context_dim}, ffn={context_dim*4}")
    print(f"  FiLM      : context({context_dim}) → γ,β ∈ R^{hidden_dim}")
    print(f"  Local LSTM: {num_layers} × {hidden_dim}, dropout={lstm_dropout}")
    print(f"  Aux loss  : α={AUX_ALPHA} × MSE( log(RV480/RV60) )")
    print("=" * 66)

    train_ids = splits["split_summary"]["train"]["ids"]
    val_ids   = splits["split_summary"]["val"]["ids"]
    test_ids  = splits["split_summary"]["test"]["ids"]
    print(f"Split: train={len(train_ids)} | val={len(val_ids)} | test={len(test_ids)}")

    # ── Build cluster map from hardcoded assignments ─────────────────
    cluster_map = build_cluster_map(stock_cols)

    print(f"\nBuilding datasets …")
    train_ds = TCNDualBranchDataset(df, stock_cols, cluster_map, train_ids, "train")
    val_ds   = TCNDualBranchDataset(df, stock_cols, cluster_map, val_ids,   "val")
    test_ds  = TCNDualBranchDataset(df, stock_cols, cluster_map, test_ids,  "test")

    pin = torch.cuda.is_available()
    tl  = DataLoader(train_ds, batch_size=bs, shuffle=True,
                     num_workers=workers, pin_memory=pin)
    vl  = DataLoader(val_ds,   batch_size=bs, shuffle=False,
                     num_workers=workers, pin_memory=pin)
    sl  = DataLoader(test_ds,  batch_size=bs, shuffle=False,
                     num_workers=workers, pin_memory=pin)

    x_local, x_tcn, y_main, y_aux, rv_in = next(iter(tl))
    print(f"\nSanity:")
    print(f"  x_local  {tuple(x_local.shape)}  — (B, K={n_clusters}, {LOCAL_LEN}, {N_CHANNELS})")
    print(f"  x_tcn    {tuple(x_tcn.shape)}  — (B, K={n_clusters}, {N_BUCKETS}, {TCN_IN_CHANNELS})")
    print(f"  y_main   {tuple(y_main.shape)}  | y_aux {tuple(y_aux.shape)}")
    assert x_local.shape[1] == n_clusters
    assert x_local.shape[2] == LOCAL_LEN  and x_local.shape[3] == N_CHANNELS
    assert x_tcn.shape[2]   == N_BUCKETS  and x_tcn.shape[3]   == TCN_IN_CHANNELS
    assert torch.isfinite(x_local).all()
    assert torch.isfinite(x_tcn).all()
    assert torch.isfinite(y_main).all()

    model = TransformerFiLMVolatilityModel(
        n_clusters          = n_clusters,
        hidden_dim          = hidden_dim,
        num_layers          = num_layers,
        dropout             = lstm_dropout,
        context_dim         = context_dim,
        transformer_heads   = transformer_heads,
        transformer_layers  = transformer_layers,
        transformer_dropout = transformer_dropout,
    )

    if pretrained_path:
        model.load_state_dict(torch.load(pretrained_path, map_location="cpu"))
        print(f"  Loaded weights from {pretrained_path}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    lstm_p   = sum(p.numel() for p in model.lstm.parameters())
    trans_p  = sum(p.numel() for p in model.transformer.parameters())
    film_p   = sum(p.numel() for p in model.film.parameters())
    head_p   = sum(p.numel() for p in model.head.parameters())
    print(f"\nModel: TransformerFiLMVolatilityModel")
    print(f"  LSTM branch       : {lstm_p:>8,} params")
    print(f"  Transformer branch: {trans_p:>8,} params")
    print(f"  FiLM layer        : {film_p:>8,} params")
    print(f"  Head + aux        : {head_p:>8,} params")
    print(f"  TOTAL             : {n_params:>8,} params\n")

    print("Training …")
    model = train_model(
        model, tl, vl,
        epochs=epochs, lr=lr, patience=patience,
        loss_type=loss_type, ckpt_path=ckpt_path,
    )

    print("\nSmearing factors (estimated on train set):")
    delta = compute_smearing_factor(model, tl)

    val_metrics  = evaluate_model(model, vl, delta,
                                  "Validation — smearing vs raw")
    test_metrics = evaluate_model(model, sl, delta,
                                  "TEST SET   — smearing vs raw  [touched once]")

    return {
        "model":        model,
        "cluster_map":  cluster_map,
        "smearing":     delta,
        "val_metrics":  val_metrics,
        "test_metrics": test_metrics,
        "n_clusters":   n_clusters,
    }


# ─────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────

def main(
    df:                   pd.DataFrame,
    epochs:               int   = 100,
    bs:                   int   = 32,
    lr:                   float = 1e-3,
    patience:             int   = 10,
    loss_type:            str   = "qlike",
    lstm_dropout:         float = 0.2,
    transformer_dropout:  float = 0.1,
    hidden_dim:           int   = 256,
    context_dim:          int   = 64,
    num_layers:           int   = 3,
    transformer_heads:    int   = 4,
    transformer_layers:   int   = 2,
    seed:                 int   = 42,
    val_ratio:            float = 0.15,
    test_ratio:           float = 0.15,
    workers:              int   = 0,
    ckpt_path:            str   = "best_transformer_lstm.pt",
) -> dict:

    print("Loading data …")
    stock_cols = [
        c for c in df.columns
        if c not in ("time_id", "seconds_in_bucket")
    ]
    print(f"  shape: {df.shape} | stocks: {len(stock_cols)} | "
          f"tids: {df['time_id'].nunique()}")

    splits = make_splits_from_time_id(df, val_ratio, test_ratio, seed)

    results = run_transformer_film_pipeline(
        df, stock_cols, splits,
        seed                = seed,
        hidden_dim          = hidden_dim,
        num_layers          = num_layers,
        lstm_dropout        = lstm_dropout,
        context_dim         = context_dim,
        transformer_heads   = transformer_heads,
        transformer_layers  = transformer_layers,
        transformer_dropout = transformer_dropout,
        epochs              = epochs,
        bs                  = bs,
        lr                  = lr,
        patience            = patience,
        loss_type           = loss_type,
        workers             = workers,
        ckpt_path           = ckpt_path,
    )

    tm_raw = results["test_metrics"]["no_smearing"]
    tm_sme = results["test_metrics"]["smearing"]

    print("\n" + "=" * 60)
    print("Pipeline complete — Test Set Summary")
    print("=" * 60)
    print(f"  {'Metric':<10}  {'No Smearing':>14}  {'Smearing':>14}")
    print(f"  {'-'*10}  {'-'*14}  {'-'*14}")
    for key in ("qlike", "rmse", "rmspe", "mape", "r2"):
        print(f"  {key.upper():<10}  "
              f"{tm_raw[key]:>14.6f}  {tm_sme[key]:>14.6f}")
    print("=" * 60)

    return results


if __name__ == "__main__":
    # df must already be in scope with columns:
    #   time_id, seconds_in_bucket, <stock_0>, <stock_1>, …
    # Stock column order must match the indices in HARDCODED_CLUSTERS.
    main(
        df,
        loss_type           = "mse",
        epochs              = 100,
        bs                  = 32,
        hidden_dim          = 256,
        context_dim         = 64,
        num_layers          = 3,
        transformer_heads   = 4,
        transformer_layers  = 2,
        lstm_dropout        = 0.2,
        transformer_dropout = 0.1,
    )

Loading data …
  shape: (2298000, 114) | stocks: 112 | tids: 3830
Transformer-FiLM + Local LSTM Volatility Pipeline
  Context   : 480s → 48 buckets of 10s each
  Transformer: layers=2, heads=4, d_model=64, ffn=256
  FiLM      : context(64) → γ,β ∈ R^256
  Local LSTM: 3 × 256, dropout=0.2
  Aux loss  : α=0.2 × MSE( log(RV480/RV60) )
Split: train=2682 | val=574 | test=574
Step 1 — Building stock profiles (train only) …
  Profiles: (112, 5)  (2682 train time_ids)
  Elbow → k=7

Step 2 — K-Means (k=7) …
  Cluster sizes: {0: 16, 1: 6, 2: 19, 3: 20, 4: 17, 5: 15, 6: 19}

Building datasets …
  [train]  2682 time_ids × 1 windows = 2682 samples | cache 720.9 MB
  [val  ]   574 time_ids × 1 windows = 574 samples | cache 154.3 MB
  [test ]   574 time_ids × 1 windows = 574 samples | cache 154.3 MB

Sanity:
  x_local  (32, 7, 60, 2)  — (B, K, 60, 2)
  x_tcn    (32, 7, 48, 3)  — (B, K, 48, 3)
  y_main   (32, 7)  | y_aux (32, 7)

Model: TransformerFiLMVolatilityModel
  LSTM branch       : 1,318,912 p